# 多模態系列 04 - ASR 自動語音辨識 (Whisper) 與語音任務管線

> HuggingFace 繁中 Cookbook 2026 版 | 模態：音訊 (Audio)

## 本 notebook 的定位

到目前為止，01~04 模組處理的都是**純文字**模態：tokenizer 把字串切成 token、model 吃 token id、evaluate 算文字指標。這份教材把同一套抽象延伸到**音訊**：你會發現「換湯不換藥」——`AutoProcessor` 一樣是前處理入口，只是它的內部分支從 tokenizer 變成 **feature_extractor**（負責把波形轉成 log-mel spectrogram），而 Whisper 本身是一個 **seq2seq (encoder-decoder)** 模型，解碼端仍然是你熟悉的語言模型。

音訊是補齊多模態拼圖的關鍵一塊：在 RAG、客服、會議記錄等真實場景，**語音→文字 (ASR)** 幾乎都是第一道前置處理，後面才接上你已經會的文字分類、摘要、檢索。

## 學習目標

1. 理解音訊的數位表示：取樣率 (sampling rate)、波形、log-mel spectrogram
2. 掌握 `WhisperProcessor` 的雙分支：`feature_extractor` + `tokenizer`
3. 用 `datasets` 的 `Audio` feature 載入並重採樣語音資料集
4. 用 `pipeline` 做長音訊 chunking 轉錄與時間戳 (timestamps)
5. 多語：`transcribe` vs `translate`，中文輸出
6. 語音評測指標：**WER / CER**（補齊 inventory 缺的語音指標）
7. 底層 `generate()` 寫法：`language`、`task` 與 generation config
8. **跨模態管線**：Whisper 轉錄 → 餵入文字分類 / 摘要模型
9. 架構對照：Whisper (seq2seq) vs Wav2Vec2 (CTC)
10. 推論加速：faster-whisper / flash attention

## 前置知識

- 已完成 `../../01-Component/01pipeline/01.pipeline.ipynb`（pipeline 抽象）
- 已完成 `../../01-Component/02tokenizer/`（tokenizer 概念，本份對照 feature_extractor）
- 已完成 `../../01-Component/05evaluate/05 evaluate.ipynb`（evaluate.load 的用法，本份新增 WER/CER）
- 跨模態管線會回收：
  - 文字分類：`../../01-Component/01pipeline/01.pipeline.ipynb`
  - 文字摘要：`../../02-Adv-tasks/07-text_summarization/summarization.ipynb`
  - 檢索問答（延伸閱讀，音訊→RAG 的銜接）：`../../02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb`

## 建議模型（2026，首選列最前）

| 模型 | 用途 | 備註 |
|---|---|---|
| `openai/whisper-large-v3-turbo` | **2026 首選**，速度快 | 解碼層精簡，品質接近 large-v3 |
| `openai/whisper-large-v3` | 最高品質基準 | VRAM 需求較大 |
| `Systran/faster-whisper-large-v3` | 推論加速對照 | CTranslate2 後端，非 transformers |
| `facebook/wav2vec2-base` | 架構對照 | CTC 架構，非 seq2seq |
| `openai/whisper-small` | 輕量替代 | 無 GPU 時用這個跑全流程 |

## 0. 版本鎖定與環境

**WHY**：跨模態套件對版本特別敏感——音訊解碼需要 `soundfile`/`librosa`，`datasets` 的 `Audio` feature 在 3.0 後 API 才穩定。把版本鎖在最前面，避免「在我電腦上可以跑」的災難。這是全 repo 一致的慣例，你在 04 模組已經看過同樣的鎖定 cell。

> 註：本 notebook 設計為**不執行**。若要實跑，取消下方安裝指令的註解。GPU 非必要，但 large-v3 級模型沒有 GPU 會非常慢，無 GPU 請改用 `whisper-small`。

In [ ]:
# Pin versions (2026 convention, consistent across the whole repo).
# Uncomment to actually install in a fresh environment.
# %pip install -q \
#     "transformers>=4.46" \
#     "datasets>=3.0" \
#     "evaluate>=0.4" \
#     "accelerate>=1.0" \
#     "soundfile" "librosa" \
#     "jiwer"  # jiwer is the backend for WER/CER metrics
# Optional, only for the acceleration section:
# %pip install -q "faster-whisper>=1.0"

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 on GPU, float32 on CPU (bf16 on CPU is slow / unsupported on many machines)
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print(f"device={DEVICE}, dtype={DTYPE}")

## 1. 音訊基礎：取樣率、log-mel spectrogram、WhisperProcessor 的雙分支

### 為什麼音訊要先講「取樣率」

聲音是連續的氣壓波，電腦無法存連續訊號，必須**取樣 (sampling)**：每秒切 N 個點。N 就是取樣率 (sampling rate)。Whisper 全系列**只吃 16kHz**（每秒 16000 個樣本）。這是最常見的踩雷點：如果你的音檔是 44.1kHz（CD 音質）直接餵進去，模型聽到的會是「變速變調」的雜訊。所以**重採樣到 16kHz 是音訊版的「對齊 tokenizer」**——就像文字一定要用對應模型的 tokenizer，音訊一定要用對應模型的取樣率。

### feature_extractor 是音訊版的 tokenizer

在純文字世界：`字串 --tokenizer--> token id --model-->`。

在 Whisper 世界：`波形 (16kHz) --feature_extractor--> log-mel spectrogram --encoder-->`。

`WhisperProcessor` 把兩件事包成一個物件：
- `processor.feature_extractor`：波形 → log-mel（80 或 128 個 mel 頻帶 × 時間幀）
- `processor.tokenizer`：模型輸出的 token id → 文字（解碼端仍是語言模型，所以還是要 tokenizer）

這正是 `AutoProcessor` 抽象的威力：**同一個 `processor` 物件，input 端管音訊、output 端管文字**。

In [ ]:
from transformers import AutoProcessor
import numpy as np

# Primary 2026 model. Swap to "openai/whisper-small" if you have no GPU.
MODEL_ID = "openai/whisper-large-v3-turbo"

processor = AutoProcessor.from_pretrained(MODEL_ID)

# The processor has two branches, mirroring tokenizer (text) vs feature_extractor (audio).
print("feature_extractor:", type(processor.feature_extractor).__name__)
print("tokenizer        :", type(processor.tokenizer).__name__)

# Whisper REQUIRES this sampling rate. Memorise it.
SR = processor.feature_extractor.sampling_rate  # -> 16000
print("required sampling rate:", SR)
print("n_mels (mel bands):", processor.feature_extractor.feature_size)

### 親手把波形變成 log-mel

**WHY**：與其相信文件，不如看 shape。我們造一段假波形（1 秒 16kHz 的正弦波），丟進 feature_extractor，觀察輸出張量的形狀。Whisper 的 feature_extractor 會自動把輸入 **pad/truncate 到固定 30 秒**（這是 Whisper encoder 的固定接收窗），所以不論你給幾秒，log-mel 的時間維度永遠是 3000 幀。理解這點，第 3 步的「30 秒 chunking」就會非常直覺。

In [ ]:
# A synthetic 1-second 16kHz sine wave (440 Hz, A4). Real audio comes in section 2.
t = np.linspace(0, 1, SR, endpoint=False)
fake_waveform = 0.1 * np.sin(2 * np.pi * 440 * t).astype(np.float32)

features = processor.feature_extractor(
    fake_waveform,
    sampling_rate=SR,
    return_tensors="pt",
)

# Shape: (batch, n_mels, n_frames). n_frames is fixed at 3000 (30s window), even for 1s input.
print("input_features shape:", features["input_features"].shape)
print("-> (batch, n_mels, time_frames); time is padded to the fixed 30s receptive window")

## 2. 載入音訊資料集：datasets 的 Audio feature 與 cast_column

**WHY**：手刻波形只能教學，真實工作要載入資料集。`datasets` 的 `Audio` feature 是音訊版的「自動解碼器」——它**懶載入 (lazy)**：在你 index 到某一筆之前不會真的去解碼音檔，省記憶體。

關鍵動作是 `cast_column(..., Audio(sampling_rate=16000))`：這行就是**強制重採樣**。你不必手動呼叫 librosa，cast 之後每次取出的 `audio["array"]` 都已經是 16kHz。這對應到純文字流程裡你用 `dataset.map(tokenize_fn)` 做前處理——只是音訊把重採樣這件事內建進 feature 型別裡了。

我們用 Common Voice 中文子集 (`zh-TW`)。它在 HF Hub 上是 gated dataset，需要先到頁面同意授權並 `huggingface-cli login`。若無法存取，下方提供公開的 fallback 資料集。

In [ ]:
from datasets import load_dataset, Audio

# --- Option A: Common Voice Chinese (Traditional). Gated: requires HF login + accepting terms. ---
# ds = load_dataset(
#     "mozilla-foundation/common_voice_17_0",
#     "zh-TW",
#     split="test",
#     streaming=False,
# )

# --- Option B (fallback, no gating): a small public multilingual ASR set for smoke-testing. ---
# google/fleurs has clean per-language splits; cmn_hans_cn is Mandarin Chinese.
ds = load_dataset("google/fleurs", "cmn_hans_cn", split="test", streaming=False)

# The single most important line for audio: force-resample every clip to 16kHz on access.
ds = ds.cast_column("audio", Audio(sampling_rate=SR))

print(ds)
print("columns:", ds.column_names)

### 看一筆資料長什麼樣

**WHY**：`Audio` feature 解碼後給你一個 dict：`array`（numpy 波形）、`sampling_rate`（應為 16000）、`path`。這就是後面所有步驟的原料。先確認 sampling_rate 真的是 16000，再往下走——99% 的 ASR 垃圾輸出都源自取樣率沒對齊。

In [ ]:
sample = ds[0]
audio = sample["audio"]

print("sampling_rate:", audio["sampling_rate"])        # must be 16000 after cast_column
print("waveform dtype/shape:", audio["array"].dtype, audio["array"].shape)
print("duration (s):", len(audio["array"]) / audio["sampling_rate"])
# fleurs ground-truth transcription column is 'transcription'; common_voice uses 'sentence'.
ref_col = "transcription" if "transcription" in ds.column_names else "sentence"
print("reference text:", sample[ref_col])

# Optional: listen to it inside Jupyter.
# from IPython.display import Audio as PlayAudio
# PlayAudio(audio["array"], rate=audio["sampling_rate"])

## 3. 用 pipeline 做長音訊轉錄：chunking 與時間戳

**WHY**：第 1 步我們看到 Whisper encoder 的接收窗固定 30 秒。那超過 30 秒的會議錄音怎麼辦？答案是 **chunking**：`pipeline` 自動把長音訊切成 30 秒一段、分別轉錄、再用 stride 重疊區拼回完整文字。你不必自己切——這正是 pipeline 抽象幫你扛掉的工程細節，跟 01 模組文字 pipeline 幫你扛掉 batching 是同一個哲學。

- `chunk_length_s=30`：每段長度，對齊 Whisper 接收窗
- `return_timestamps=True`：回傳每段文字的起訖秒數（做字幕、定位用）
- `batch_size`：多段平行送 GPU，加速長音訊

> VRAM 警告：`whisper-large-v3-turbo` 約需 2~3GB（bf16）。若 OOM，把 `MODEL_ID` 換成 `openai/whisper-small`（約 1GB）或降 `batch_size`。

In [ ]:
from transformers import pipeline

asr = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_ID,
    torch_dtype=DTYPE,
    device=DEVICE,
    # The model and its processor (feature_extractor + tokenizer) are wired up automatically.
)

result = asr(
    sample["audio"]["array"],
    chunk_length_s=30,        # split long audio into 30s windows (Whisper's fixed receptive field)
    stride_length_s=5,        # overlap between chunks so words on boundaries aren't lost
    batch_size=8,             # process multiple chunks in parallel
    return_timestamps=True,   # get per-segment (start, end) seconds
)

print("full text:", result["text"])
print("\nsegments with timestamps:")
for chunk in result["chunks"]:
    ts = chunk["timestamp"]   # (start_sec, end_sec)
    print(f"  [{ts[0]:>6.2f} - {ts[1]:>6.2f}]  {chunk['text']}")

### word-level 時間戳

**WHY**：`return_timestamps=True` 給的是「段落級」時間戳。做卡拉OK字幕、逐字對齊時需要**逐字 (word-level)** 時間戳，傳 `return_timestamps="word"` 即可。代價是稍慢，因為要解 cross-attention 對齊。

In [ ]:
word_result = asr(
    sample["audio"]["array"],
    chunk_length_s=30,
    return_timestamps="word",   # word-level alignment instead of segment-level
)
for chunk in word_result["chunks"][:10]:
    print(chunk["timestamp"], chunk["text"])

## 4. 多語控制：language 與 task（transcribe vs translate）

**WHY**：Whisper 是多語模型。它**會自動偵測語言**，但自動偵測在短音訊或混語時容易猜錯。生產環境若已知語言，**明確指定 `language` 永遠比讓它猜更穩**——這是少數「多給資訊就更好」的旋鈕。

兩個核心參數：
- `language`：來源語言（如 `"chinese"`、`"english"`、`"japanese"`）
- `task`：
  - `"transcribe"`：原語言轉文字（中文音 → 中文字）
  - `"translate"`：**直接翻成英文**（中文音 → 英文字），Whisper 內建的零樣本語音翻譯能力

在 `pipeline` 裡，這兩個參數透過 `generate_kwargs` 傳給底層 `generate()`。

In [ ]:
# Transcribe: Chinese audio -> Chinese text (force language, don't let it guess).
zh_text = asr(
    sample["audio"]["array"],
    generate_kwargs={"language": "chinese", "task": "transcribe"},
)["text"]
print("transcribe (zh -> zh):", zh_text)

# Translate: Chinese audio -> English text, in one shot, no separate MT model needed.
en_text = asr(
    sample["audio"]["array"],
    generate_kwargs={"language": "chinese", "task": "translate"},
)["text"]
print("translate  (zh -> en):", en_text)

## 5. 評測：WER 與 CER

**WHY**：在純文字分類你用 accuracy/F1（見 `../../01-Component/05evaluate/`）。ASR 的標準指標不同，inventory 之前缺這塊，這裡補齊：

- **WER (Word Error Rate)**：以「詞」為單位的編輯距離 / 參考詞數。適合英文等有空白分詞的語言。
- **CER (Character Error Rate)**：以「字元」為單位。**中文沒有空白分詞，WER 不適用，中文 ASR 一律看 CER**。這是中文語音評測的關鍵差異。

兩者都是 `evaluate.load(...)`，介面跟你用過的文字指標完全一致——又一次印證「換模態不換抽象」。數值越低越好（0 = 完美）。

> 中文評測前通常會做正規化：去標點、繁簡統一、全形半形統一。否則「，」「。」會被算成錯誤，虛高 CER。下方示範最小正規化。

In [ ]:
import evaluate
import re

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def normalize_zh(text: str) -> str:
    # Minimal normalization for Chinese ASR scoring: strip punctuation & whitespace.
    # Production setups also do traditional<->simplified and full<->half width unification.
    text = re.sub(r"[\s,.!?;:　，。！？；：]", "", text)
    return text.strip()

# Evaluate Whisper over a small slice of the dataset.
N = 10  # keep small for a demo; raise for a real eval
predictions, references = [], []
for item in ds.select(range(N)):
    pred = asr(
        item["audio"]["array"],
        generate_kwargs={"language": "chinese", "task": "transcribe"},
    )["text"]
    predictions.append(normalize_zh(pred))
    references.append(normalize_zh(item[ref_col]))

# For Chinese, CER is the meaningful number; WER is shown for contrast only.
cer = cer_metric.compute(predictions=predictions, references=references)
print(f"CER (use this for Chinese): {cer:.3f}")
# WER on un-segmented Chinese is misleading; included to make the point explicit.
wer = wer_metric.compute(predictions=predictions, references=references)
print(f"WER (misleading for Chinese without word segmentation): {wer:.3f}")

## 6. 底層 generate() 寫法，對照 pipeline

**WHY**：`pipeline` 很好用，但它藏住了真正發生的事。當你要**微調 Whisper、客製 decoding、或 debug 為什麼輸出語言不對**時，必須會手寫底層流程。這一步把 pipeline 拆開成三段，正好對應第 1 步講的抽象鏈：

```
feature_extractor(波形) -> input_features -> model.generate() -> token ids -> tokenizer.decode()
```

這跟你在純文字 seq2seq（如 02 摘要模型）手寫 `model.generate()` 是**完全一樣的流程**，唯一差別是輸入端從 `input_ids` 換成 `input_features`。

### generate() 的語言與任務控制

語言和任務直接以具名參數傳入 `generate(language=..., task=...)`，Whisper 的 generation config 會在內部把這兩個值轉成對應的解碼器提示 token，你不需要手動處理任何 token 操作。

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",   # 2026 convention: let accelerate place the model
)
model.eval()

# Step 1: feature_extractor turns waveform -> log-mel input_features (this is the audio 'tokenizing').
inputs = processor(
    sample["audio"]["array"],
    sampling_rate=SR,
    return_tensors="pt",
)
input_features = inputs["input_features"].to(model.device, dtype=DTYPE)

# Step 2: generate. Pass language and task directly; the generation config handles prompt tokens.
with torch.no_grad():
    generated_ids = model.generate(
        input_features,
        language="chinese",
        task="transcribe",
        max_new_tokens=256,
        num_beams=1,            # greedy; raise for quality at the cost of speed
    )

# Step 3: tokenizer decodes ids -> text (same as any text seq2seq model).
text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("low-level generate() output:", text)

## 7. 跨模態管線：Whisper 轉錄 → 文字模型分類 / 摘要

**WHY**：這是整份 notebook 的核心價值，也是「多模態」的真正意義——**音訊不是終點，是文字模型的新輸入源**。你在 01~02 模組學的文字分類、摘要能力，現在可以套在「任何說出來的話」上：客服錄音情緒分析、會議錄音自動摘要、語音指令意圖分類。

管線結構非常乾淨：
```
音訊 --Whisper(ASR)--> 文字 --既有文字模型--> 結果
```

第二段我們**直接重用 01/02 學過的 pipeline**，不需要任何新概念。這就是模組化的回報：把語音「降維」成文字後，下游全部是你已經會的東西。

In [ ]:
# Stage 1: speech -> text (reuse our ASR pipeline from section 3).
transcript = asr(
    sample["audio"]["array"],
    generate_kwargs={"language": "chinese", "task": "transcribe"},
)["text"]
print("[ASR transcript]", transcript)

# Stage 2a: text classification (the skill from 01-Component/01pipeline).
# Multilingual sentiment model so it works on Chinese transcripts.
classifier = pipeline(
    task="text-classification",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    device=DEVICE,
)
print("[sentiment]", classifier(transcript))

### 接上摘要模型（回收 02 模組）

**WHY**：會議/演講錄音通常很長，轉錄出的逐字稿沒人想讀。把 ASR 的輸出餵進摘要模型（對應 `../../02-Adv-tasks/07-text_summarization/summarization.ipynb` 學的 seq2seq 摘要），就得到「錄音 → 重點摘要」的端到端產品。注意：Whisper 本身也是 seq2seq，摘要模型也是 seq2seq——你現在是把**兩個 encoder-decoder 串起來**，只是中間經過了文字這個共同介面。

In [ ]:
# Stage 2b: summarization (the skill from 02-Adv-tasks/07-text_summarization).
# mT5/multilingual model handles Chinese; swap to your fine-tuned summarizer from module 02.
summarizer = pipeline(
    task="summarization",
    model="csebuetnlp/mT5_multilingual_XLSum",
    device=DEVICE,
)

# In practice, transcript here would be a long meeting/lecture, not a single sentence.
summary = summarizer(transcript, max_length=60, min_length=10, do_sample=False)
print("[summary]", summary[0]["summary_text"])

# Takeaway: once audio is reduced to text, the ENTIRE text toolbox from modules 01-02 applies
# unchanged. This text interface is also the bridge to audio -> RAG (see next-steps section).

## 8. 架構對照：Whisper (seq2seq) vs Wav2Vec2 (CTC)

**WHY**：ASR 有兩大流派，理解差異才知道何時選哪個。這是「對照模型加深理解」的環節，跟你之前比較 cross-encoder vs dual-encoder（04 模組）是同樣的學習策略。

| 面向 | Whisper | Wav2Vec2 |
|---|---|---|
| 架構 | **seq2seq** (encoder-decoder) | **CTC** (encoder + linear head) |
| 解碼 | 自回歸生成 token，**有語言模型** | 逐幀預測字元，**無自回歸** |
| 標點/大小寫 | 內建，輸出自帶標點 | 通常無標點、全小寫 |
| 多語/翻譯 | 原生支援 99 語 + translate | 單語為主，需各語言 fine-tune |
| 速度 | 較慢（自回歸） | **很快**（一次前向） |
| 幻覺 | 會（生成式通病，靜音段可能瞎掰） | 幾乎不會（無生成） |
| 適用 | 通用轉錄、多語、要標點 | 即時、低延遲、單一領域 |

**一句話**：Whisper 把 ASR 當「翻譯題」（聲音→文字的生成），Wav2Vec2 把它當「逐幀分類題」（每個時間幀貼一個字元標籤，再用 CTC 規則合併重複與 blank）。生成式給你流暢標點但可能幻覺；CTC 又快又老實但要自己加標點。

In [ ]:
# Wav2Vec2 (CTC) for contrast. Note: it does NOT use generate(); it does a single forward
# pass and we argmax over per-frame logits, then CTC-decode.
from transformers import AutoModelForCTC, AutoProcessor as CTCAutoProcessor

W2V_ID = "facebook/wav2vec2-base-960h"  # English CTC model (base has no LM head fine-tune for zh)

ctc_processor = CTCAutoProcessor.from_pretrained(W2V_ID)
ctc_model = AutoModelForCTC.from_pretrained(W2V_ID).to(DEVICE)
ctc_model.eval()

# Same feature_extractor abstraction, different model family.
ctc_inputs = ctc_processor(
    sample["audio"]["array"],
    sampling_rate=SR,
    return_tensors="pt",
).input_values.to(DEVICE)

with torch.no_grad():
    logits = ctc_model(ctc_inputs).logits     # (batch, frames, vocab) -- per-frame, NOT generated

predicted_ids = torch.argmax(logits, dim=-1)  # greedy per frame
ctc_text = ctc_processor.batch_decode(predicted_ids)[0]  # CTC collapse happens inside decode
print("Wav2Vec2 (CTC) output:", ctc_text)
print("-> notice: no punctuation, all caps/lower, single forward pass (no autoregression)")

## 9. 推論加速選項：faster-whisper 與 flash attention

**WHY**：large-v3 在生產環境的瓶頸是**解碼速度**（自回歸天生慢）。有兩條主流加速路線，依你的部署環境選擇。

### 路線 A：留在 transformers，開 flash attention / SDPA

最小改動：`attn_implementation="flash_attention_2"`（需裝 `flash-attn`，僅 Ampere+ GPU）或 `"sdpa"`（PyTorch 內建，幾乎到處可用）。再搭配 `whisper-large-v3-turbo`（解碼層更薄）本身就比 large-v3 快數倍。

In [ ]:
# Route A: faster decoding while staying in transformers.
fast_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
    # "flash_attention_2" needs the flash-attn package + Ampere(or newer) GPU.
    # "sdpa" is the safe default: built into PyTorch, broad hardware support.
    attn_implementation="sdpa",
)
fast_model.eval()
print("loaded with attn_implementation=sdpa")
# Combine with whisper-large-v3-turbo (thinner decoder) for the best transformers-native speed.

### 路線 B：換後端到 faster-whisper (CTranslate2)

**WHY**：`faster-whisper` 不是 transformers，是基於 CTranslate2 的重寫，做了 INT8 量化、kernel 優化，**CPU 上也能快 4 倍以上**，記憶體更省。代價是離開了 transformers 生態（不能直接 fine-tune、API 不同）。**部署選 faster-whisper，研究/微調留在 transformers**——這是很典型的「訓練用 HF、上線換推論引擎」分工，呼應 inventory 提到的 faster-whisper 加速對照。

In [ ]:
# Route B: faster-whisper (CTranslate2 backend). Different API, not a transformers model.
# %pip install -q faster-whisper
from faster_whisper import WhisperModel

# compute_type="int8" on CPU, "float16"/"bfloat16" on GPU.
fw_model = WhisperModel(
    "large-v3",
    device=DEVICE,
    compute_type="int8" if DEVICE == "cpu" else "float16",
)

# It streams segments (a generator) instead of returning one dict.
segments, info = fw_model.transcribe(
    sample["audio"]["array"],
    language="zh",
    task="transcribe",
    beam_size=5,
)
print("detected language:", info.language, "prob:", round(info.language_probability, 3))
for seg in segments:
    print(f"[{seg.start:.2f}-{seg.end:.2f}] {seg.text}")

## 10. 小結、練習題與下一站

### 核心收穫：音訊也只是 processor 抽象

這份 notebook 從頭到尾在證明一件事：**換模態不換心法**。

| 純文字（01~04 你已學） | 音訊（本份） |
|---|---|
| `tokenizer` 字串→token | `feature_extractor` 波形→log-mel |
| 對齊 tokenizer | 對齊 16kHz 取樣率 |
| `dataset.map(tokenize)` | `cast_column(Audio(16000))` |
| `pipeline('text-classification')` | `pipeline('automatic-speech-recognition')` |
| accuracy / F1 | **WER / CER**（中文用 CER） |
| seq2seq `model.generate(input_ids=)` | seq2seq `model.generate(input_features=)` |

`AutoProcessor` 是統一入口：input 端依模態切換 (feature_extractor / tokenizer / image_processor)，output 端仍回到文字。一旦把語音「降維」成文字，**整個 01~02 的文字工具箱原封不動就能用**——這就是第 7 步跨模態管線的全部魔法。

### 重點提醒（生產踩雷清單）

- 取樣率不對 = 垃圾輸出，永遠先確認 16kHz。
- 中文評測用 **CER 不用 WER**，且評測前要正規化標點/繁簡。
- 已知語言就明確傳 `language=`，別讓模型猜。
- Whisper 會在靜音/雜訊段**幻覺**；高可靠場景需加 VAD 前處理或門檻過濾。
- 部署要快 → faster-whisper / turbo + sdpa；研究微調 → 留在 transformers。

### 練習題

1. **CER vs WER 實證**：拿一段中文音訊，分別算 CER 和「先用 jieba 斷詞再算的 WER」，比較哪個更貼近你主觀感受的錯誤率。
2. **chunk 邊界實驗**：找一段 > 60 秒、句子橫跨 30 秒邊界的音訊，比較 `stride_length_s=0` 與 `=5` 的輸出差異，觀察邊界漏字。
3. **translate 評測**：用 `task="translate"` 把中文音訊轉英文，並用 `sacrebleu`（`evaluate.load('sacrebleu')`）對照人工英譯，思考「語音翻譯」該用什麼指標。
4. **架構選型**：給定「即時直播字幕（低延遲、英文、容忍無標點）」需求，論證該選 Whisper 還是 Wav2Vec2，並說明理由。
5. **端到端產品**：串接 `Whisper → 摘要 → 文字分類`，做一個「錄音 → 重點摘要 + 情緒標籤」的小函式，輸入一段音訊回傳 dict。
6. **微調前奏**：用本份的 `feature_extractor` + `cast_column` 把 Common Voice 中文做成可訓練的 `input_features` / `labels`，為下一步 Whisper LoRA 微調鋪路（解碼端 tokenizer 一樣，回收 03-PEFT 的技巧）。

### 下一站

- **音訊 → RAG**：把本份的 ASR 當前置，串到 `../../02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb`，做「語音提問 → 檢索 → 回答」的語音版 RAG。文字介面就是音訊與檢索之間的橋。
- **Whisper 微調 (LoRA)**：把 03-PEFT 的 LoRA 套在 Whisper 上做領域口音/術語適配，訓練流程用 `Trainer` + `Seq2SeqTrainingArguments`（`bf16=True, save_safetensors=True, seed=42, warmup_ratio=0.1, lr_scheduler_type='cosine'`），結尾 `push_to_hub` 與 model card——與 04 模組訓練慣例完全一致。
- **更廣的多模態**：本系列 05-Multimodal 的影像分支（CLIP/SigLIP、VLM）用的是同一個 `AutoProcessor` 抽象，只是 input 端換成 image_processor。你現在已經掌握把這套心法套到任何模態的通用模式。